<a href="https://colab.research.google.com/github/F1ratbl/bmu1417-lab/blob/main/hafta01.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import sys, pandas as pd
print(sys.version.split()[0], pd.__version__)

3.13.15 2.2.3


In [18]:
import requests, pandas as pd

url = "https://servisnet.afad.gov.tr/apigateway/deprem/apiv2/event/filter"
params = {"start": "2000-09-01T00:00:00", "end": "2026-09-27T00:00:00", "minmag": 2.0}
veri = requests.get(url, params=params, timeout=30).json()

df = pd.DataFrame(veri)
df["magnitude"] = pd.to_numeric(df["magnitude"])
print(df.shape)
df.head(3)[["date", "magnitude", "location"]]

(268103, 15)


,date,magnitude,location
0,2005-08-20T02:21:26.76,3.0,Ege Denizi - [08.50 km] Datça (Muğla)
1,2005-08-20T06:09:48.39,3.5,Marmara Denizi - [06.39 km] Şarköy (Tekirdağ)
2,2005-08-16T23:41:36.87,3.3,Ege Denizi - [15.95 km] Karaburun (İzmir)


In [19]:
df.groupby("province")["magnitude"].agg(["count", "max"]) \
  .sort_values("count", ascending=False).head(100)

,count,max
province,,
Muğla,34181,6.5
İzmir,19330,6.6
Kütahya,17874,5.7
Van,16604,6.7
Kahramanmaraş,14385,7.7
...,...,...
Bartın,145,4.8
Kilis,111,3.8
Iğdır,94,4.8


In [20]:
import os
df.to_csv("deprem.csv", index=False)
df.to_parquet("deprem.parquet", index=False)

for f in ["deprem.csv", "deprem.parquet"]:
    print(f, round(os.path.getsize(f) / 1024, 1), "KB")

deprem.csv 33490.6 KB
deprem.parquet 10687.3 KB


In [23]:
import plotly.express as px

buyuk = df[df["magnitude"] >= 5]

fig = px.scatter_map(
    buyuk,
    lat="latitude",
    lon="longitude",
    size="magnitude",
    color="magnitude",
    color_continuous_scale="YlOrRd",
    size_max=15,
    opacity=0.7,
    hover_name="location",
    hover_data={"date": True, "magnitude": True, "depth": True,
                "latitude": False, "longitude": False},
    center={"lat": 39.0, "lon": 35.0},
    zoom=4.5,
    map_style="carto-positron",
    height=600,
    title=f"5 ve üstü depremler ({len(buyuk)} adet)",
)
fig.show()